# Migrate Hybrid Search from Chroma Cloud to Qdrant Cloud

This notebook accompanies the tutorial at https://qdrant.tech/documentation/tutorials-operations/chroma-hybrid-retrieval-migration/

You start from a Chroma Cloud collection that serves hybrid search: dense vectors from Chroma's default embedding function, BM25 sparse vectors, and metadata filters. You move it to Qdrant Cloud, replace Chroma's embedding functions with Qdrant Cloud Inference, and translate the fused query.

Chroma Cloud stops accepting writes on **December 7, 2026** and stops serving reads on **December 22, 2026**.

You need:

- A Chroma Cloud account with an API key, tenant, and database.
- A Qdrant Cloud cluster on version 1.20 or newer.
- Docker on your own machine, for the migration tool.

# 1. Create a Qdrant Cluster

If you do not already have a Qdrant cluster, follow these steps to create one:

- Register for a [Qdrant Cloud account](https://cloud.qdrant.io/) using your email, Google, or GitHub credentials.
- Under Create a Free Cluster, enter a cluster name and select your preferred cloud provider and region.
- Click Create Free Cluster.
- Copy the API key when prompted and store it somewhere safe, as it won't be displayed again.
- Copy the Cluster Endpoint. It should look something like `https://xxx.cloud.qdrant.io`.

# 2. Set Up the Client Connections

Install the Qdrant client and the Chroma client. `snowballstemmer` is required by Chroma's BM25 embedding function.

In [1]:
%pip install -q "qdrant-client>=1.20.0" "chromadb>=1.5.9" snowballstemmer

Add these as Colab secrets: `QDRANT_URL`, `QDRANT_API_KEY`, `CHROMA_API_KEY`, `CHROMA_TENANT`, and `CHROMA_DATABASE`. Outside Colab, set them as environment variables.

In [2]:
import os

try:
    from google.colab import userdata

    get_secret = userdata.get
except ImportError:
    get_secret = os.environ.get

QDRANT_URL = get_secret("QDRANT_URL")
QDRANT_API_KEY = get_secret("QDRANT_API_KEY")
CHROMA_API_KEY = get_secret("CHROMA_API_KEY")
CHROMA_TENANT = get_secret("CHROMA_TENANT")
CHROMA_DATABASE = get_secret("CHROMA_DATABASE")

COLLECTION = "atlas-support"

In [3]:
import chromadb
from qdrant_client import QdrantClient, models

chroma = chromadb.CloudClient(
    tenant=CHROMA_TENANT,
    database=CHROMA_DATABASE,
    api_key=CHROMA_API_KEY,
)

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    cloud_inference=True,
)
client.info()

VersionInfo(title='qdrant - vector search engine', version='1.20.0', commit='222b2e44ab1c3f17e8246396ead6558e45041df7')

The `cloud_inference=True` parameter enables [Qdrant Cloud Inference](https://qdrant.tech/documentation/embeddings/qdrant-cloud-inference/), so the cluster generates vectors from text you send. Step 5 uses it to replace Chroma's embedding functions.

# 3. The Example Chroma Collection

This notebook needs a Chroma Cloud collection to migrate. The next two cells create an example one, so you can run every step: a support assistant for a fictional product called Atlas, with dense vectors, BM25 sparse vectors, and metadata for filtering and citations. It stands in for the collection your application already uses.

**Migrating your own collection?** Set `COLLECTION` to its name, skip the next two cells, and adapt the field names, filters, and baseline queries in the rest of the notebook to match your data.

In the example data, the two authentication errors have near-identical wording and different remedies, so only an exact identifier match separates them. The three backup records are close paraphrases distinguished by metadata alone. Versions are stored as integer major and minor numbers, so filters can select a range such as version 2 and later.

In [4]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Record:
    chroma_id: str
    document: str
    product: str
    version_major: int
    version_minor: int
    article_id: str


CORPUS: list[Record] = [
    Record(
        "atlas-2-session",
        "Atlas 2.0 ends your session after 30 minutes of inactivity.",
        "atlas",
        2,
        0,
        "session-expiration",
    ),
    Record(
        "atlas-2-auth4017",
        "AUTH4017 means your Atlas API key has expired. Create a new key in Settings > API Keys.",
        "atlas",
        2,
        0,
        "auth4017",
    ),
    Record(
        "atlas-2-auth4018",
        "AUTH4018 means your Atlas API key lacks permission for this operation.",
        "atlas",
        2,
        0,
        "auth4018",
    ),
    Record(
        "atlas-2-backup",
        "To restore an Atlas 2.0 backup, open Settings > Backups, select a backup, and choose Restore.",
        "atlas",
        2,
        0,
        "restore-backup",
    ),
    Record(
        "atlas-1-backup",
        "To restore an Atlas 1.0 backup, open Admin > Recovery, upload the backup file, and choose Import.",
        "atlas",
        1,
        0,
        "restore-backup",
    ),
    Record(
        "boreal-2-backup",
        "To restore a Boreal 2.0 backup, open Workspace > Snapshots, select a snapshot, and choose Recover.",
        "boreal",
        2,
        0,
        "restore-backup",
    ),
]

The example collection uses Chroma's default embedding function for dense vectors, with cosine distance, and `ChromaBm25EmbeddingFunction` for a sparse vector index on the `sparse_embedding` key. Chroma generates both from the document text. Chroma Cloud accepts writes only until December 7, 2026, so this cell works only before then.

In [5]:
from chromadb import K, Schema, SparseVectorIndexConfig, VectorIndexConfig
from chromadb.utils.embedding_functions import (
    ChromaBm25EmbeddingFunction,
    DefaultEmbeddingFunction,
)

schema = Schema()
schema.create_index(
    config=VectorIndexConfig(space="cosine", embedding_function=DefaultEmbeddingFunction())
)
schema.create_index(
    config=SparseVectorIndexConfig(
        source_key=K.DOCUMENT,
        embedding_function=ChromaBm25EmbeddingFunction(),
        bm25=True,
    ),
    key="sparse_embedding",
)

collection = chroma.get_or_create_collection(name=COLLECTION, schema=schema)
collection.upsert(
    ids=[r.chroma_id for r in CORPUS],
    documents=[r.document for r in CORPUS],
    metadatas=[
        {
            "product": r.product,
            "version_major": r.version_major,
            "version_minor": r.version_minor,
            "article_id": r.article_id,
        }
        for r in CORPUS
    ],
)
collection.count()

6

## Record the Chroma Baseline

Before migrating, capture Chroma's results for a set of queries that covers your retrieval behavior: dense matches, lexical matches, metadata filters, and a filter with no matching records. For your own collection, replace these queries with ones from your application. The hybrid query is the one the application runs today, and `default=1000` keeps documents that only one branch finds.

In [6]:
from chromadb import Knn, Rrf, Search


def chroma_search(question: str, where) -> list[str]:
    search = (
        Search()
        .where(where)
        .rank(
            Rrf(
                ranks=[
                    Knn(query=question, return_rank=True, limit=20, default=1000),
                    Knn(
                        query=question,
                        key="sparse_embedding",
                        return_rank=True,
                        limit=20,
                        default=1000,
                    ),
                ],
                k=60,
            )
        )
        .limit(5)
        .select(K.SCORE)
    )
    return [row["id"] for row in collection.search(search).rows()[0]]


def chroma_version(major: int, minor: int):
    return (K("product") == "atlas") & (K("version_major") == major) & (K("version_minor") == minor)


SIGN_IN = "Why do I have to sign in again after leaving the app idle?"
AUTH = "How do I resolve AUTH4017?"
BACKUP = "How do I restore a backup?"

CHROMA_BASELINE = {
    ("2.0", SIGN_IN): chroma_search(SIGN_IN, chroma_version(2, 0)),
    ("2.0", AUTH): chroma_search(AUTH, chroma_version(2, 0)),
    ("2.0", BACKUP): chroma_search(BACKUP, chroma_version(2, 0)),
    ("1.0", BACKUP): chroma_search(BACKUP, chroma_version(1, 0)),
    ("2+", BACKUP): chroma_search(BACKUP, (K("product") == "atlas") & (K("version_major") >= 2)),
    ("9.0", BACKUP): chroma_search(BACKUP, chroma_version(9, 0)),
}
for (version, question), ids in CHROMA_BASELINE.items():
    print(version, question, ids)

2.0 Why do I have to sign in again after leaving the app idle? ['atlas-2-session', 'atlas-2-auth4017', 'atlas-2-auth4018', 'atlas-2-backup']
2.0 How do I resolve AUTH4017? ['atlas-2-auth4017', 'atlas-2-session', 'atlas-2-auth4018', 'atlas-2-backup']
2.0 How do I restore a backup? ['atlas-2-backup', 'atlas-2-auth4017', 'atlas-2-session', 'atlas-2-auth4018']
1.0 How do I restore a backup? ['atlas-1-backup']
2+ How do I restore a backup? ['atlas-2-backup', 'atlas-2-auth4017', 'atlas-2-session', 'atlas-2-auth4018']
9.0 How do I restore a backup? []


# 4. Migrate the Collection

Run the migration tool from a terminal on your own machine, with your credentials filled in. The Qdrant URL is the gRPC endpoint on port 6334.

```bash
docker run --rm registry.cloud.qdrant.io/library/qdrant-migration chroma \
  --chroma.url 'https://api.trychroma.com' \
  --chroma.tenant 'your-tenant-id' \
  --chroma.database 'your-database' \
  --chroma.collection 'atlas-support' \
  --chroma.auth-type token \
  --chroma.token-header X-Chroma-Token \
  --chroma.token 'your-chroma-api-key' \
  --qdrant.url 'https://your-cluster.cloud.qdrant.io:6334' \
  --qdrant.api-key 'your-qdrant-api-key' \
  --qdrant.collection 'atlas-support' \
  --qdrant.document-field document \
  --qdrant.id-field __id__ \
  --migration.batch-size 300
```

- `--qdrant.document-field` is the payload field that receives the document text.
- `--qdrant.id-field` keeps the original Chroma ID. Qdrant point IDs must be integers or UUIDs, so the tool derives one and stores the original here.
- The tool reads the distance metric from the Chroma collection. `--qdrant.distance-metric` overrides it.

The tool migrates indexed sparse vectors automatically, using each Chroma metadata key as the Qdrant vector name. For indexes configured for BM25, it also enables the IDF modifier. A sparse field without a sparse vector index is skipped with a warning.

Point IDs are derived from the Chroma document ID, so if a second run targets the same Qdrant collection, any document ID present in both sources resolves to the same point and the later run overwrites it.

## Verify the Transfer

Compare the exact point count with the Chroma collection, and confirm that the sparse vector is configured on the collection and present on the points.

In [7]:
info = client.get_collection(COLLECTION)
print("Chroma records:", collection.count())
print("Qdrant points: ", client.count(COLLECTION, exact=True).count)
print("dense:", info.config.params.vectors)
print("sparse:", info.config.params.sparse_vectors)

records, _ = client.scroll(COLLECTION, limit=3, with_payload=True, with_vectors=True)
for record in records:
    print(record.payload["__id__"], sorted(record.vector.keys()))

Chroma records: 6
Qdrant points:  6
dense: size=384 distance=<Distance.COSINE: 'Cosine'> hnsw_config=None quantization_config=None on_disk=None memory=None datatype=None multivector_config=None
sparse: {'sparse_embedding': SparseVectorParams(index=None, modifier=<Modifier.IDF: 'idf'>)}
atlas-2-session ['', 'sparse_embedding']
atlas-1-backup ['', 'sparse_embedding']
boreal-2-backup ['', 'sparse_embedding']


## Index the Filter Fields

The migration tool does not create payload indexes. Create one for each field your filters use: keyword indexes for the product and article fields, and integer indexes, which support range filters, for the version numbers. Qdrant builds them from the payload already in the collection.

In [8]:
for field_name, field_schema in (
    ("product", models.PayloadSchemaType.KEYWORD),
    ("article_id", models.PayloadSchemaType.KEYWORD),
    ("version_major", models.PayloadSchemaType.INTEGER),
    ("version_minor", models.PayloadSchemaType.INTEGER),
):
    client.create_payload_index(
        collection_name=COLLECTION,
        field_name=field_name,
        field_schema=field_schema,
        wait=True,
    )

# 5. Replace Chroma's Embedding Functions

The migration copies vectors, but not the embedding functions configured on the Chroma collection. Keep generating vectors for queries and new documents with the same models.

Chroma's default embedding function runs `all-MiniLM-L6-v2`, and `ChromaBm25EmbeddingFunction` produces the same vectors as Qdrant's BM25 on English text at its default settings. Both are free hosted models on Qdrant Cloud Inference. The tutorial maps other Chroma embedding functions, such as Chroma Cloud Qwen and Chroma Cloud SPLADE.

In [9]:
DENSE_MODEL = "sentence-transformers/all-minilm-l6-v2"
SPARSE_MODEL = "qdrant/bm25"

# 6. Translate the Hybrid Query

Each Chroma branch becomes a prefetch with the same candidate limit. A filter on the query applies to every prefetch, so both branches search only the requested product and versions, as with Chroma's `where`. `MatchValue` translates equality conditions, and `Range` translates `>=` and the other comparisons. Set `k=60` explicitly, since Qdrant's default is `2`.

In [10]:
def hybrid_search(question: str, scope: models.Filter) -> list[models.ScoredPoint]:
    query_vector = models.Document(text=question, model=DENSE_MODEL)
    sparse_query = models.Document(text=question, model=SPARSE_MODEL)
    result = client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(query=query_vector, limit=20),
            models.Prefetch(query=sparse_query, using="sparse_embedding", limit=20),
        ],
        query=models.RrfQuery(rrf=models.Rrf(k=60)),
        query_filter=scope,
        limit=5,
        with_payload=True,
    )
    return result.points


def match(key: str, value: str | int) -> models.FieldCondition:
    return models.FieldCondition(key=key, match=models.MatchValue(value=value))


def atlas_version(major: int, minor: int) -> models.Filter:
    return models.Filter(
        must=[
            match("product", "atlas"),
            match("version_major", major),
            match("version_minor", minor),
        ]
    )


atlas_2_and_later = models.Filter(
    must=[
        match("product", "atlas"),
        models.FieldCondition(key="version_major", range=models.Range(gte=2)),
    ]
)

# 7. Verify Retrieval Behavior

Run the baseline queries against Qdrant. Confirm that each question returns its expected chunk first, no result comes from a product or version the filter excludes, and the absent-version query returns nothing. The `chroma:` line under each check shows the baseline for comparison. Use the labeled expectations, not Chroma's ranking, to judge relevance.

In [11]:
CHECKS = [
    # (baseline key, filter, expected top result, chunks the filter must exclude)
    (("2.0", SIGN_IN), atlas_version(2, 0), "atlas-2-session", ()),
    (("2.0", AUTH), atlas_version(2, 0), "atlas-2-auth4017", ()),
    (("2.0", BACKUP), atlas_version(2, 0), "atlas-2-backup", ("atlas-1-backup", "boreal-2-backup")),
    (("1.0", BACKUP), atlas_version(1, 0), "atlas-1-backup", ("atlas-2-backup", "boreal-2-backup")),
    (("2+", BACKUP), atlas_2_and_later, "atlas-2-backup", ("atlas-1-backup", "boreal-2-backup")),
    (("9.0", BACKUP), atlas_version(9, 0), None, ()),
]

for key, scope, expected, excluded in CHECKS:
    ids = [point.payload["__id__"] for point in hybrid_search(key[1], scope)]
    top_ok = ids[:1] == [expected] if expected else not ids
    passed = top_ok and not set(excluded) & set(ids)
    print("PASS" if passed else "FAIL", key[0], key[1])
    print("  qdrant:", ids)
    print("  chroma:", CHROMA_BASELINE[key])

PASS 2.0 Why do I have to sign in again after leaving the app idle?
  qdrant: ['atlas-2-session', 'atlas-2-auth4017', 'atlas-2-auth4018', 'atlas-2-backup']
  chroma: ['atlas-2-session', 'atlas-2-auth4017', 'atlas-2-auth4018', 'atlas-2-backup']
PASS 2.0 How do I resolve AUTH4017?
  qdrant: ['atlas-2-auth4017', 'atlas-2-auth4018', 'atlas-2-session', 'atlas-2-backup']
  chroma: ['atlas-2-auth4017', 'atlas-2-session', 'atlas-2-auth4018', 'atlas-2-backup']
PASS 2.0 How do I restore a backup?
  qdrant: ['atlas-2-backup', 'atlas-2-auth4017', 'atlas-2-auth4018', 'atlas-2-session']
  chroma: ['atlas-2-backup', 'atlas-2-auth4017', 'atlas-2-session', 'atlas-2-auth4018']
PASS 1.0 How do I restore a backup?
  qdrant: ['atlas-1-backup']
  chroma: ['atlas-1-backup']
PASS 2+ How do I restore a backup?
  qdrant: ['atlas-2-backup', 'atlas-2-auth4017', 'atlas-2-auth4018', 'atlas-2-session']
  chroma: ['atlas-2-backup', 'atlas-2-auth4017', 'atlas-2-session', 'atlas-2-auth4018']
PASS 9.0 How do I restore a

If an expected result is missing, test dense and sparse retrieval separately before adjusting candidate limits or fusion settings.

In [12]:
def single_branch(question: str, dense: bool) -> list[str]:
    result = client.query_points(
        collection_name=COLLECTION,
        query=models.Document(text=question, model=DENSE_MODEL if dense else SPARSE_MODEL),
        using=None if dense else "sparse_embedding",
        query_filter=atlas_version(2, 0),
        limit=3,
        with_payload=True,
    )
    return [point.payload["__id__"] for point in result.points]


print("dense :", single_branch(AUTH, dense=True))
print("sparse:", single_branch(AUTH, dense=False))

dense : ['atlas-2-auth4017', 'atlas-2-auth4018', 'atlas-2-session']
sparse: ['atlas-2-auth4017']


The dense branch returns AUTH4018 close behind AUTH4017, because the two sentences differ by one digit and one clause. The sparse branch matches the literal token. That separation is why the sparse branch matters: a migration that silently lost it could still pass a spot-check on this question.

# 8. Move the Write Path

Before December 7, send inserts, updates, and deletions to Qdrant instead of Chroma.

For every write, reproduce the migration tool's point-ID conversion: preserve valid UUIDs and convert other Chroma IDs to UUIDv5 using the URL namespace. An update then targets the migrated record instead of creating another point. Keep the vector names and payload fields consistent with the migrated collection, and regenerate both vectors when document text changes. `upload_points` batches the writes and retries failed batches.

In [13]:
import uuid


def point_id_for(chroma_id: str) -> str:
    try:
        return str(uuid.UUID(chroma_id))
    except ValueError:
        return str(uuid.uuid5(uuid.NAMESPACE_URL, chroma_id))


def write_chunks(records: list[Record]) -> None:
    """Write or replace chunks, both vectors and payload together, in retried batches."""
    client.upload_points(
        collection_name=COLLECTION,
        points=[
            models.PointStruct(
                id=point_id_for(record.chroma_id),
                vector={
                    "": models.Document(text=record.document, model=DENSE_MODEL),
                    "sparse_embedding": models.Document(text=record.document, model=SPARSE_MODEL),
                },
                payload={
                    "document": record.document,
                    "__id__": record.chroma_id,
                    "product": record.product,
                    "version_major": record.version_major,
                    "version_minor": record.version_minor,
                    "article_id": record.article_id,
                },
            )
            for record in records
        ],
        max_retries=3,
        wait=True,
    )


def delete_article(article_id: str) -> None:
    """Remove every chunk of an article, however many there are."""
    client.delete(
        collection_name=COLLECTION,
        points_selector=models.FilterSelector(
            filter=models.Filter(
                must=[
                    models.FieldCondition(
                        key="article_id", match=models.MatchValue(value=article_id)
                    )
                ]
            )
        ),
        wait=True,
    )


def point_count() -> int:
    return client.count(COLLECTION, exact=True).count

A migrated record keeps its point ID, so updating it does not create a duplicate:

In [14]:
before = point_count()
write_chunks([CORPUS[0]])
print("migrated record rewritten:", point_count() == before)

migrated record rewritten: True


Add, revise, and delete a record. Running the same write twice must not duplicate. When you re-chunk an article, delete its old chunks with a filter on `article_id` first, since the new chunk count may differ.

In [15]:
new_chunk = Record(
    "atlas-2-ratelimit",
    "RATE429 means you exceeded the Atlas request quota.",
    "atlas",
    2,
    0,
    "rate429",
)

write_chunks([new_chunk])
write_chunks([new_chunk])  # a retry must not duplicate
print("after add:", point_count())
print(
    "found:",
    [
        point.payload["__id__"]
        for point in hybrid_search("What does RATE429 mean?", atlas_version(2, 0))
    ],
)

revised = Record(
    new_chunk.chroma_id,
    "RATE429 means you exceeded the Atlas request quota. Retry after 60 seconds.",
    "atlas",
    2,
    0,
    "rate429",
)
write_chunks([revised])
print("after revise:", point_count())

delete_article("rate429")
print("after delete:", point_count())

after add: 7
found: ['atlas-2-ratelimit', 'atlas-2-auth4018', 'atlas-2-auth4017', 'atlas-2-session', 'atlas-2-backup']
after revise: 7
after delete: 6


# 9. Cut Over

Once retrieval matches your baseline and the write path is tested, switch reads to Qdrant. Chroma Cloud serves reads until December 22, 2026, so keep your Chroma collection and recorded baseline until then, and re-run the migration before that date if you need to.

To remove the example collections afterward, uncomment and run the next cell.

In [ ]:
# chroma.delete_collection(COLLECTION)
# client.delete_collection(COLLECTION)